### Data augmentation

- Apply geometrical transformations to bad and bad_masks images
- Apply photometric transformations to bad images and keep unchange the bad_masks
- Do not apply any transformation to good images
  

In [10]:
from pathlib import Path
import shutil, random, numpy as np
from skimage.io import imread, imsave

factor_of_augmentation = 16
dataset ="carpet"
src = Path(f"../datasets/{dataset}")
dst = Path(f"{src.parent}/{dataset}-da{factor_of_augmentation}")

seed = 1234; random.seed(seed); np.random.seed(seed)

if dst.exists(): shutil.rmtree(dst)
dst.mkdir(parents=True, exist_ok=True)

good_src = Path(f"{src}/good")
good_dst = Path(f"{dst}/good")
shutil.copytree(good_src, good_dst, dirs_exist_ok=True)

bad_dst = Path(f"{dst}/bad")
bad_masks_dst = Path(f"{dst}/bad-masks")
bad_dst.mkdir(parents=True, exist_ok=True)
bad_masks_dst.mkdir(parents=True, exist_ok=True)

print('Source path:', dst)
print('Destination path:', dst)

Source path: ../datasets/carpet-da16
Destination path: ../datasets/carpet-da16


In [11]:
from random import random as rndr, uniform as rndu, choice as rndc
from skimage.transform import rotate, AffineTransform, warp, swirl
from skimage.exposure import adjust_gamma
from skimage.filters import gaussian, unsharp_mask, rank
from skimage.util import random_noise
from skimage.color import rgb2hsv, hsv2rgb
from skimage.morphology import disk
from scipy.ndimage import map_coordinates, gaussian_filter

def aug_params(h, w):
    return {
        "hflip":        {"on": rndr() < 0.5},
        "vflip":        {"on": rndr() < 0.5},
        "rotate":       {"on": rndr() < 0, "angle": rndu(-2, 2)},
        "translate_x":  {"on": rndr() < 0, "tx": rndu(-0.05, 0.05) * w},
        "translate_y":  {"on": rndr() < 0, "ty": rndu(-0.05, 0.05) * h},
        "scale":        {"on": rndr() < 0, "value": rndu(0.95, 1.05)},
        "shear":        {"on": rndr() < 0.5, "value": rndu(-0.08, 0.08)},
        "brightness":   {"on": rndr() < 0.5, "gain": rndu(0.9, 1.1)},
        "contrast":     {"on": rndr() < 0, "gain": rndu(0.9, 1.1)},
        "gamma":        {"on": rndr() < 0.5, "value": rndu(0.9, 1.1)},
        "blur":         {"on": rndr() < 0.5, "sigma": rndu(0.4, 1.2)},
        "gaussiannoise":{"on": rndr() < 0.5, "var": rndu(0.0005, 0.003)},
        "salt_pepper":  {"on": rndr() < 0, "amount": rndu(0.002, 0.01)},
        "speckle_noise":{"on": rndr() < 0, "var": rndu(0.0005, 0.002)},
        "hue":          {"on": rndr() < 0, "delta": rndu(-0.3, 0.3)},
        "saturation":   {"on": rndr() < 0, "gain": rndu(0.9, 1.1)},
        "value":        {"on": rndr() < 0, "gain": rndu(0.9, 1.1)},
        "rgb_gain":     {"on": rndr() < 0, "r": rndu(0.9, 1.1), "g": rndu(0.9, 1.1),"b": rndu(0.9, 1.1)},
        "rank_median":  {"on": rndr() < 0, "radius": rndc([1, 2])},
        "rank_mean":    {"on": rndr() < 0, "radius": rndc([1, 2])},
        "rank_maximum": {"on": rndr() < 0, "radius": rndc([1, 2])},
        "rank_minimum": {"on": rndr() < 0, "radius": rndc([1, 2])},
        "swirl":        {"on": rndr() < 0.5, "strength": rndu(0.5, 1.0), "radius": rndu(20, min(h, w) * 0.25)},
        "elastic":      {"on": rndr() < 0, "alpha": rndu(4, 12), "sigma": rndu(4, 8)}
    }

def rank_rgb(x, fn, radius):
    fp = disk(radius)
    if x.ndim == 2: return fn(x, fp)
    return np.stack([fn(x[..., c], fp) for c in range(x.shape[2])], axis=-1)

def elastic_deform(x, alpha, sigma, order):
    h, w = x.shape[:2]
    dx = gaussian_filter((np.random.rand(h, w) * 2 - 1), sigma, mode="reflect") * alpha
    dy = gaussian_filter((np.random.rand(h, w) * 2 - 1), sigma, mode="reflect") * alpha
    xx, yy = np.meshgrid(np.arange(w), np.arange(h))
    indices = np.reshape(yy + dy, (-1, 1)), np.reshape(xx + dx, (-1, 1))
    if x.ndim == 2: y = map_coordinates(x, indices, order=order, mode="reflect").reshape(h, w)
    else: y = np.stack([map_coordinates(x[..., c], indices, order=order, mode="reflect").reshape(h, w) for c in range(x.shape[2])], axis=-1)
    return y

def clip(x): return np.clip(x,0,255).astype(np.uint8)

def apply_geom(x,p,order):
    y=x
    tx=p["translate_x"]["tx"] if p["translate_x"]["on"] else 0
    ty=p["translate_y"]["ty"] if p["translate_y"]["on"] else 0
    sc=p["scale"]["value"] if p["scale"]["on"] else 1
    sh=p["shear"]["value"] if p["shear"]["on"] else 0
    if tx!=0 or ty!=0 or sc!=1 or sh!=0: y=warp(y, AffineTransform(scale=(sc,sc), translation=(tx,ty), shear=sh).inverse, preserve_range=True, order=order, mode="constant", cval=0)
    if p["hflip"]["on"]:    y=np.fliplr(y)
    if p["vflip"]["on"]:    y=np.flipud(y)
    if p["rotate"]["on"]:   y=rotate(y, p["rotate"]["angle"], resize=False, preserve_range=True, order=order, mode="constant", cval=0)
    if p["swirl"]["on"]:    y=swirl(y, strength=p["swirl"]["strength"], radius=p["swirl"]["radius"], preserve_range=True, order=order)
    if p["elastic"]["on"]:  y=elastic_deform(y, p["elastic"]["alpha"], p["elastic"]["sigma"], order=order)
    return clip(y)

def apply_photo_img(x,p):
    y=x.astype(np.uint8); m=y.mean(axis=(0,1),keepdims=True)
    if p["brightness"]["on"]:   y=clip(y.astype(np.float32)*p["brightness"]["gain"])
    if p["contrast"]["on"]:     y=clip((y.astype(np.float32)-m)*p["contrast"]["gain"]+m)
    if p["gamma"]["on"]:        y=clip(255*adjust_gamma(y/255.0,gamma=p["gamma"]["value"]))
    if p["blur"]["on"]:         y=clip(gaussian(y,sigma=p["blur"]["sigma"],preserve_range=True,channel_axis=-1))
    if p["gaussiannoise"]["on"]:y=clip(255*random_noise(y/255.0,mode="gaussian",var=p["gaussiannoise"]["var"]))
    if p["salt_pepper"]["on"]:  y=clip(255*random_noise(y/255.0,mode="s&p",amount=p["salt_pepper"]["amount"]))
    if p["speckle_noise"]["on"]:y=clip(255*random_noise(y/255.0,mode="speckle",var=p["speckle_noise"]["var"]))
    if p["rank_median"]["on"]:  y=rank_rgb(y,rank.median,p["rank_median"]["radius"]).astype(np.uint8)
    if p["rank_mean"]["on"]:    y=rank_rgb(y,rank.mean,p["rank_mean"]["radius"]).astype(np.uint8)
    if p["rank_maximum"]["on"]: y=rank_rgb(y,rank.maximum,p["rank_maximum"]["radius"]).astype(np.uint8)
    if p["rank_minimum"]["on"]: y=rank_rgb(y,rank.minimum,p["rank_minimum"]["radius"]).astype(np.uint8)
    if p["rgb_gain"]["on"]: 
        z=y.astype(np.float32); 
        z[...,0]*=p["rgb_gain"]["r"]; 
        z[...,1]*=p["rgb_gain"]["g"]; 
        z[...,2]*=p["rgb_gain"]["b"]; y=clip(z)
    if p["hue"]["on"] or p["saturation"]["on"] or p["value"]["on"]:
        z=rgb2hsv(y/255.0)
        if p["hue"]["on"]: z[...,0]=(z[...,0]+p["hue"]["delta"])%1.0
        if p["saturation"]["on"]: z[...,1]=np.clip(z[...,1]*p["saturation"]["gain"],0,1)
        if p["value"]["on"]: z[...,2]=np.clip(z[...,2]*p["value"]["gain"],0,1)
        y=clip(255*hsv2rgb(z))
    return y

p = aug_params(100, 100)
for k, v in p.items():
    if v.get("on"):
        params = {kk: vv for kk, vv in v.items() if kk != "on"}; print(k, params)



vflip {}
shear {'value': 0.046203634757805354}
brightness {'gain': 1.0246562950078337}
gamma {'value': 0.9228825939377375}
blur {'sigma': 0.7894012324838067}


In [12]:
bad_root = Path(f"{src}/bad")
mask_root = Path(f"{src}/bad-masks")

for defect_dir in bad_root.iterdir():
    if not defect_dir.is_dir(): continue
    defect = defect_dir.name
    out_b = Path(f"{dst}/bad/{defect}")
    out_m = Path(f"{dst}/bad-masks/{defect}")
    out_b.mkdir(parents=True, exist_ok=True)
    out_m.mkdir(parents=True, exist_ok=True)
    for img_path in sorted(defect_dir.glob("*.png")):
        mask_path = Path(f"{mask_root}/{defect}/{img_path.name}")
        print(img_path)
        if not mask_path.exists(): continue
        img = imread(img_path)
        msk = imread(mask_path)
        imsave(Path(f"{out_b}/{img_path.name}"), img)
        imsave(Path(f"{out_m}/{mask_path.name}"), msk)
        h, w = img.shape[0], img.shape[1]
        for k in range(1, factor_of_augmentation + 1):
            p = aug_params(h, w)
            img2 = apply_photo_img(apply_geom(img, p, order=1), p)
            msk2 = apply_geom(msk, p, order=0)
            name = f"{img_path.stem}-da{k}.png"
            imsave(Path(f"{out_b}/{name}"), img2)
            imsave(Path(f"{out_m}/{name}"), msk2)
print("'Done!")
            

../datasets/carpet/bad/color/000.png
../datasets/carpet/bad/color/001.png
../datasets/carpet/bad/color/002.png
../datasets/carpet/bad/color/003.png
../datasets/carpet/bad/color/004.png
../datasets/carpet/bad/color/005.png
../datasets/carpet/bad/color/006.png
../datasets/carpet/bad/color/007.png
../datasets/carpet/bad/color/008.png
../datasets/carpet/bad/color/009.png
../datasets/carpet/bad/color/010.png
../datasets/carpet/bad/color/011.png


/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/011.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/011-da1.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/011-da2.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/011-da3.png is a low contrast image
  return func(*args, **

../datasets/carpet/bad/color/012.png


/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/012.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/012-da1.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/012-da2.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/color/012-da3.png is a low contrast image
  return func(*args, **

../datasets/carpet/bad/color/013.png


...

../datasets/carpet/bad/thread/018.png


/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/thread/018-da1.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/thread/018-da2.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/thread/018-da3.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/thread/018-da4.png is a low contrast image
  return func(*

../datasets/carpet/bad/hole/000.png
../datasets/carpet/bad/hole/001.png


/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/hole/001.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/hole/001-da1.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/hole/001-da2.png is a low contrast image
  return func(*args, **kwargs)
/Users/dsage/miniconda3/envs/dl/lib/python3.10/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/dsage/Desktop/defect-detection-kaggle/datasets/carpet-da16/bad-masks/hole/001-da3.png is a low contrast image
  return func(*args, **kwar

../datasets/carpet/bad/hole/002.png
../datasets/carpet/bad/hole/003.png
../datasets/carpet/bad/hole/004.png
../datasets/carpet/bad/hole/005.png
../datasets/carpet/bad/hole/006.png
../datasets/carpet/bad/hole/007.png
../datasets/carpet/bad/hole/008.png
../datasets/carpet/bad/hole/009.png
../datasets/carpet/bad/hole/010.png
../datasets/carpet/bad/hole/011.png
../datasets/carpet/bad/hole/012.png
../datasets/carpet/bad/hole/013.png
../datasets/carpet/bad/hole/014.png
../datasets/carpet/bad/hole/015.png
../datasets/carpet/bad/hole/016.png
'Done!
